# Week 1 Activity — Broken Poetry (Text Is Just Numbers)

## What's this all about?

Here's a claim that sounds wild the first time you hear it: **a computer has never seen a letter.** Not an "A," not a smiley face, not a single word of the text message you sent this morning. As far as the machine is concerned, there's no such thing as text. There are only numbers.

When you type the letter `A`, your computer doesn't store an `A`. It stores the *number* **65**. The letter `B` is **66**. A space is **32**. Every character you can type has a number, called its *code point*, and the agreed-upon list of which character gets which number is **Unicode**. For plain English letters, digits, and punctuation, Unicode's numbers are the same ones the older **ASCII** code used, which is why you'll see ASCII come up a lot. Your texts, this sentence, the entire internet are all just sequences of numbers.

In this *computational notebook*, a kind of living document that can have text (like this) and running code side by side, we'll take a piece of text, watch it turn into numbers, do some weird math to those numbers, and then turn them back into text. The result is going to be a kind of jagged broken poetry.

You will **not** write any code today. You'll *run* code and *change a few settings* to see what happens. The whole point is to *watch* the text become numbers and back again.

> **Running this:** click a gray code cell and press **Shift+Enter** (or the ▶ button). Go top to bottom. To change what happens, edit the boxes on the right (in Colab) or the values in quotes, then re-run the cell.

## Step 1: Type some text

Edit the line below. Put in your own sentence, a song lyric, your name, whatever you like. Then run the cell. (Later on, come back and try a name with an accent, like `José`, or an emoji. Do you think their numbers will look like the others?)

In [ ]:
text = "Good dogs make the best computers."  #@param {type:"string"}

print("Your text:", text)
print("It's made of", len(text), "characters.")

## Step 2: Watch it become numbers

This is the big reveal. The function `ord()` ("ordinal") tells you the number behind any character. So from the computer's perspective, here's what you wrote:

In [ ]:
numbers = [ord(character) for character in text]

print("Your text, as the computer actually stores it:")
print(numbers)

There, that's your sentence as a list of code points. Find a number you can decode: spaces are `32`, capital letters run `65`–`90`, lowercase letters run `97`–`122`. Can you spot the spaces in your sentence?

And since every number is stored as bits, we can peek one more layer down. This cell shows the first few characters as numbers and then as binary:

In [ ]:
for character in text[:5]:
    print(repr(character), ord(character), format(ord(character), "08b"))

Each plain English character fits in eight bits, with a number no bigger than 127. If you put an accented letter in Step 1, you'll see a number past 127, beyond the end of ASCII. An emoji's number is so big that it needs a lot more than eight bits, because emoji live much further out in Unicode.

Now let's go backwards. The function `chr()` is the reverse of `ord()`, so if you give it a number, you get the character back:

In [ ]:
back_to_text = "".join(chr(n) for n in numbers)

print("And turning the numbers back into text:")
print(back_to_text)

As one would hope, the process of changing representation was completely reversible! Now that we can go text → numbers → text, we can play around *in the middle state*, while it's numbers.

## Step 3: Shift every number (a secret-code classic)

What if we added the same amount to every number before turning it back into text? `A` (65) plus 1 becomes `B` (66). The whole message slides down the alphabet. This is a version of the **Caesar cipher**, a 2,000-year-old secret code, and it's just *addition*.

Change the `shift` value and re-run to see different results:

In [ ]:
shift = 1  #@param {type:"slider", min:-25, max:25, step:1}

shifted_numbers = [n + shift for n in numbers]
shifted_text = "".join(chr(n) for n in shifted_numbers)

print(f"Shifted by {shift}:")
print(shifted_text)
print(shifted_numbers)

Try `shift = 1`, then `2`, then `4`. The text gets scrambled into gibberish, but it's *reversible* gibberish. Run this next cell to shift everything back by the same amount:

In [ ]:
unshifted_text = "".join(chr(n - shift) for n in shifted_numbers)

print(f"Shifting by {-shift} undoes it:")
print(unshifted_text)

Now try something bigger, like `13`, or something negative, like `-3`. Did parts of your text disappear? Look at the numbers to find out why. Right after `z` (122) come `{ | } ~` (123–126), and then at 127 and for a while after that there are *control characters*, numbers that were set aside to mean things like "delete" or "ring a bell" instead of a visible symbol. The same thing happens at the bottom end: a space is 32, so any negative shift pushes it down into the invisible control codes below 32. So with `13`, every letter from `r` to `z` turns invisible. (Run the shift-back cell anyway! Everything comes back, because the numbers were never lost. They just didn't have anything to draw.)

## Step 4: Echo and repeat

Let's do something different to the numbers: this time we *repeat* them! We'll copy every number in the list a few times before turning them back into characters. This is where it starts to feel like poetry getting stuck in a hall of mirrors.

In [ ]:
echo = 3  #@param {type:"slider", min:1, max:8, step:1}

echoed_numbers = []
for n in numbers:
    echoed_numbers = echoed_numbers + [n] * echo   # n, n, n, ...

echoed_text = "".join(chr(n) for n in echoed_numbers)

print(f"Each number copied {echo} times:")
print(echoed_numbers[:30], "...")
print(echoed_text)

Crank `echo` up and down. The *spaces* echo too, because to the computer a space is just the number `32`, no different from any other.

## Step 5: Cut it up

Next up is a **cut**. Let's keep only every Nth number in the list and throw the rest away. The writer [William Burroughs](https://en.wikipedia.org/wiki/Cut-up_technique) and the Dadaists before him made art by slicing up newspapers and reassembling the scraps; we'll do it with one line.

In [ ]:
keep_every = 2  #@param {type:"slider", min:1, max:6, step:1}

cut_numbers = numbers[::keep_every]
cut_text = "".join(chr(n) for n in cut_numbers)

print(f"Keeping every number {keep_every} spots apart:")
print(cut_numbers)
print(cut_text)

With `keep_every = 2` you get every 2nd character, with `3` every 3rd, and so on.

## Step 6: The cut-up machine — make your broken poem

So far we've done the same thing to *every* character, evenly. That's neat, but it's a little tidy. Real cut-up poetry has *chaos* in it. Some words survive untouched. Some get stretched, some stutter, some vanish, some come back wrong. You can't predict which.

So for the finale we'll build a little **cut-up machine**. It walks through your text **one whole word at a time**, and for each word it *rolls the dice*: maybe it leaves the word alone, or maybe it reaches into a menu of effects and does something weird to that *entire word*. It might echo it, reverse it, shout it, or drop it on the floor. Because it's random, **every run is different**, and that unpredictability is where the poetry lives.

Two knobs:

- **`chaos`** — how *often* a word gets messed with. At `0`, your text comes out clean; at `100`, almost every word gets hit.
- **`seed`** — this sets the dice. Same seed = same poem (so you can keep a version you like). **Change the seed to re-roll** and get a totally different cut-up from the same words.

Run it, then keep nudging `seed` until something jumps out at you. You're hunting for a happy accident.

In [ ]:
import random

chaos = 60  #@param {type:"slider", min:0, max:100, step:5}
seed  = 1   #@param {type:"slider", min:0, max:50, step:1}

random.seed(seed)   # fix the dice so a given seed always gives the same poem

# Our menu of effects. Each one takes a whole word and returns a mangled version.
def echo(word):     return " ".join([word] * random.randint(2, 4))   # say it say it say it
def stutter(word):  return word[0] + "-" + word[0] + "-" + word       # b-b-broken
def reverse(word):  return word[::-1]                                  # word -> drow
def shout(word):    return word.upper() + "!"                          # LOUD
def stretch(word):  return " ".join(word)                             # s p a c e d   o u t
def shift(word):    return "".join(chr(ord(c) + 3) for c in word)     # nudged off the alphabet
def vanish(word):   return ""                                         # cut: gone entirely

menu = [echo, stutter, reverse, shout, stretch, shift, vanish]

# Walk the text word by word, rolling the dice on each one.
new_words = []
for word in text.split():
    if random.randint(1, 100) <= chaos:      # did this word's number come up?
        effect = random.choice(menu)         # if so, pick a random effect...
        new_words.append(effect(word))       # ...and apply it to the WHOLE word
    else:
        new_words.append(word)               # otherwise leave it be

# Stitch it back together with ragged line breaks, so it reads like a poem.
poem_lines, line = [], []
for piece in new_words:
    if piece == "":          # a vanished word is skipped, leaving no trace
        continue
    line.append(piece)
    if random.random() < 0.35:               # sometimes break to a new line
        poem_lines.append(" ".join(line))
        line = []
if line:
    poem_lines.append(" ".join(line))

print("YOUR BROKEN POEM")
print("=" * 40)
print("\n".join(poem_lines))
print("=" * 40)

Run it three or four times with *different seeds* and watch how the same sentence keeps coming back as a different creature. Then turn `chaos` way down (say `20`) for a poem that's mostly intact with a few glitches, or all the way up (`100`) for total static. Somewhere in there is a version that's *yours*.

## Reflection

A sentence or two each (these go in your discussion post):

1. Pick one transformation you tried (a shift, an echo, a cut, or one of the cut-up machine's effects). What did it do to the *numbers*, and how did that change the text you could read?
2. Did you get any strange symbols, or characters that vanished? Using the numbers, explain what happened to them. (If everything came out clean, try `shift = 13` and explain that instead.)
3. If you tried an accented letter or an emoji, how did its number compare to the plain English letters? Why do you think a code with only 128 slots, like ASCII, wasn't enough for everybody's names?

## What to turn in

Post on the **Week 1 discussion thread**:

1. Your favorite broken poem. Copy the output and paste it in, and tell us the `chaos` and `seed` settings that produced it (so a classmate could reproduce it).
2. One number from your Step 2 output, and which character it stood for.
3. Your reflection answers.

Then reply to a classmate whose broken poem you liked.

*Low-stakes — graded on completion. You don't have to write any code; you have to run it, play with it, and notice what's happening.*